# Random Survival Forest

The objective of this notebook is to train the first baseline survival prediction model using the processed METABRIC dataset.

Unlike conventional classification algorithms, survival models are designed to predict **both** whether an event (dead/alive in this case) occurs and the expected time (survival length) until that event occurs. In medical datasets, this is particularly important because many patients remain alive when data collection ends (censored patients). Their exact survival time is therefore unknown, resulting in **right-censored observations** that cannot be handled appropriately by standard machine learning algorithms.

To address this challenge, this project employs a **Random Survival Forest (RSF)**, an ensemble learning algorithm specifically developed for survival analysis.

The goals of this notebook are to:

- prepare survival labels,
- train a baseline Random Survival Forest,
- generate patient risk predictions,
- save the trained model for downstream evaluation and interpretation.

### A. Survival Analysis

Most machine learning models answer a single question, that is: "what will happen?" (predictions). Examples include predicting survival/fraud transactions/email spam. Survival analysis answers a more informative question: *what will happen, and when will it happen?*

Instead of predicting only the occurrence of an event, survival models estimate how the probability of survival changes over time. This makes survival analysis particularly valuable in medicine, where both the occurrence and timing of outcomes influence clinical decision-making.

### B. Censoring

Clinical studies rarely observe every patient's complete survival history. Some patients remain alive when the study concludes, while others may withdraw from follow-up before an event occurs. Although the exact survival time is unknown for these individuals, it is known that they survived for at least the duration of observation. These patients are referred to as **right-censored observations**. Rather than discarding censored patients, survival analysis incorporates this partial information directly into model training, allowing substantially more efficient use of clinical data.

### C. Algorithm Used
A Random Survival Forest extends the traditional Random Forest algorithm to survival data. Instead of predicting a class label, each decision tree estimates patient survival based on: **patterns learned from clinical variables, gene expression profiles, and mutation status.** Each tree is trained using a different subset of patients and predictor variables. The final prediction is obtained by combining the estimates from hundreds of individual trees.This ensemble approach reduces variance + enables the model to capture complex nonlinear biological relationships without assuming proportional hazards.

## 1. Imports

In [1]:
import pandas as pd

from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv

import joblib

## 2. Training

In [2]:
X_train = pd.read_csv("../data/processed/train_features.csv")
X_test = pd.read_csv("../data/processed/test_features.csv")

y_time_train = pd.read_csv("../data/processed/train_time.csv").squeeze()
y_event_train = (
    pd.read_csv("../data/processed/train_event.csv")
    .squeeze()
    .astype(bool)
)

In [3]:
leakage_cols = [
    "death_from_cancer_Living",
    "death_from_cancer_Died of Other Causes"
]

X_train = X_train.drop(
    columns=leakage_cols,
    errors="ignore"
)

X_test = X_test.drop(
    columns=leakage_cols,
    errors="ignore"
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

Training features: (1523, 722)
Testing features: (381, 722)


**Preparing Survival Labels**

Unlike standard ML algorithms, survival models require 2 target variables:

- **Survival time**, representing the duration of patient follow-up.
- **Event indicator**, specifying whether the event (death) occurred or whether the patient was censored.

The `Surv` object provided by `scikit-survival` combines these variables into a structured format suitable for model training.

In [4]:
y_train = Surv.from_arrays(
    event=y_event_train,
    time=y_time_train
)

#y_train[:5] verify

**Initializing  Model**

The baseline Random Survival Forest consists of 300 decision trees. A large ensemble generally produces more stable predictions by averaging the results of many independently trained trees. 

In [5]:
rsf = RandomSurvivalForest(

    n_estimators=300,
    max_features="sqrt",
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1

)

In [ ]:
rsf.fit(
    X_train,
    y_train
)

## 3. Predicting Patient Risk

After training, RSF assigns each patient a **risk score**. These scores do not represent probabilities! They provide a relative ranking of patients according to predicted survival risk, i.e.,:

- higher score -> higher predicted risk
- lower score -> lower predicted risk

Model performance will be evaluated quantitatively in the next nb.

In [ ]:
risk_scores = rsf.predict(
    X_test
)

#risk_scores[:10] verify

In [8]:
joblib.dump(

    rsf,

    "../models/random_survival_forest.pkl"

)
print("Model saved successfully.")

Model saved successfully.


# Summary

In this notebook, a baseline **Random Survival Forest (RSF)** model was trained using the processed METABRIC breast cancer dataset.

Unlike traditional classification models, the RSF accounts for both the occurrence and timing of patient outcomes while naturally handling right-censored observations. Survival labels were first converted into the structured format required by `scikit-survival`, after which a forest of 300 decision trees was trained on the processed clinical, mutation, and gene expression features.

The trained model successfully generated individualized risk scores for unseen patients and was saved for use in subsequent analyses.

The next stage of the project will evaluate the predictive performance of the model using survival-specific metrics and visualizations to determine how accurately it ranks patient prognosis.